## Olist Ecommerce Analytics

In [10]:
# Library import

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
from pathlib import Path

In [11]:
BASE = Path(".")
RAW = Path("../data/raw")
CLEAN = Path("../data/clean")
CLEAN.mkdir(parents=True, exist_ok=True)

In [12]:
# -----------------------------------
# Load CSVs
# -----------------------------------
orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
products = pd.read_csv(RAW / "olist_products_dataset.csv")
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")
cats = pd.read_csv(RAW / "product_category_name_translation.csv")

In [13]:
# -----------------------------------
# Standardize Columns
# -----------------------------------
for df in [orders, items, customers, products, reviews, payments, sellers, cats]:
    df.columns = df.columns.str.lower().str.strip()

# -----------------------------------
# Convert Dates
# -----------------------------------
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for c in date_cols:
    if c in orders.columns:
        orders[c] = pd.to_datetime(orders[c], errors="coerce")

In [14]:
# -----------------------------------
# Product Category Translation
# -----------------------------------
products = products.merge(
    cats,
    on="product_category_name",
    how="left"
)

In [15]:
# -----------------------------------
# Aggregate Payments (one row/order)
# -----------------------------------
pay_agg = payments.groupby("order_id", as_index=False).agg(
    total_payment_value=("payment_value", "sum"),
    max_installments=("payment_installments", "max")
)

In [16]:
# -----------------------------------
# Aggregate Reviews (one row/order)
# -----------------------------------
rev_agg = reviews.groupby("order_id", as_index=False).agg(
    review_score=("review_score", "mean")
)

In [17]:
# -----------------------------------
# Aggregate Order Items (one row/order)
# IMPORTANT: Prevent duplication
# -----------------------------------
item_agg = items.groupby("order_id", as_index=False).agg(
    total_items=("order_item_id", "count"),
    revenue=("price", "sum"),
    freight_value=("freight_value", "sum"),
    seller_count=("seller_id", "nunique")
)

# Top category per order
item_prod = items.merge(
    products[["product_id", "product_category_name_english"]],
    on="product_id",
    how="left"
)

top_cat = (
    item_prod.groupby(["order_id", "product_category_name_english"])
    .size()
    .reset_index(name="cnt")
    .sort_values(["order_id", "cnt"], ascending=[True, False])
    .drop_duplicates("order_id")
    [["order_id", "product_category_name_english"]]
)

item_agg = item_agg.merge(top_cat, on="order_id", how="left")

In [18]:
# -----------------------------------
# Build Master Table (one row/order)
# -----------------------------------
master = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(item_agg, on="order_id", how="left")
    .merge(pay_agg, on="order_id", how="left")
    .merge(rev_agg, on="order_id", how="left")
)

In [21]:
master.shape

(99441, 20)

In [22]:
# -----------------------------------
# Feature Engineering
# -----------------------------------
master["delivery_days"] = (
    master["order_delivered_customer_date"]
    - master["order_purchase_timestamp"]
).dt.days

master["approval_hours"] = (
    master["order_approved_at"]
    - master["order_purchase_timestamp"]
).dt.total_seconds() / 3600

master["purchase_month"] = master["order_purchase_timestamp"].dt.to_period("M").astype(str)

master["purchase_year"] = master["order_purchase_timestamp"].dt.year

master["is_delayed"] = np.where(
    master["order_delivered_customer_date"] >
    master["order_estimated_delivery_date"],
    1, 0
)

master["aov"] = master["revenue"]

In [23]:
master.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
total_items                       775
revenue                           775
freight_value                     775
seller_count                      775
product_category_name_english    2185
total_payment_value                 1
max_installments                    1
review_score                      768
delivery_days                    2965
approval_hours                    160
purchase_month                      0
purchase_year                       0
is_delayed                          0
aov                               775
dtype: int64

In [24]:
# Fill Nulls
master["review_score"] = master["review_score"].fillna(0)
master["delivery_days"] = master["delivery_days"].fillna(0)

In [25]:
sales_df = master[
    master["order_status"].isin(["delivered","shipped","invoiced"])
].copy()

sales_df = sales_df[sales_df["revenue"].notna()]

In [27]:
sales_df.shape

(97896, 26)